# SFT on HumanEval (Qwen2.5-Coder-3B, LoRA, T4)

Supervised fine-tuning of **Qwen2.5-Coder-3B-Instruct** on the HumanEval subset of `final_dataset_v2.csv` using **QLoRA** (4-bit + LoRA). Designed for Google Colab with **T4 GPU**.

Outputs LoRA adapters in PEFT format and a FedAvg-ready `lora_state_dict.pt`.

In [ ]:
# Check GPU (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
!pip install -q "transformers>=4.36" "peft>=0.7" "bitsandbytes>=0.41" "trl>=0.7,<0.20" "datasets" "accelerate" "pandas"
# Restart runtime after install if needed, then run the rest of the notebook.

## Data loading

In [ ]:
from google.colab import files
import pandas as pd

# Option A: Upload final_dataset_v2.csv when prompted
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print("Using CSV:", CSV_PATH)

# Option B (comment out Option A and set path manually):
# CSV_PATH = "/content/drive/MyDrive/.../final_dataset_v2.csv"  # after mounting Drive

df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))

In [ ]:
df_he = df[df["dataset"] == "humaneval"].copy()
df_he = df_he.dropna(subset=["prompt", "canonical_solution"])
df_he["canonical_solution"] = df_he["canonical_solution"].astype(str).str.strip()
df_he = df_he[df_he["canonical_solution"].str.len() > 0]
assert len(df_he) == 164, f"Expected 164 HumanEval rows, got {len(df_he)}"
print("HumanEval rows:", len(df_he))

## Parse prompt and build chat messages

In [ ]:
SEP = "\n\nUser: "

def row_to_messages(row):
    prompt_str = str(row["prompt"]).strip()
    idx = prompt_str.find(SEP)
    if idx == -1:
        import warnings
        warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
        system_content = "You are an expert Python developer. Complete the function provided by the user."
        user_content = prompt_str.replace("System: ", "", 1).strip()
    else:
        system_content = prompt_str[:idx].replace("System: ", "", 1).strip()
        user_content = prompt_str[idx + len(SEP):].strip()
    solution = str(row["canonical_solution"]).strip()
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": solution},
    ]

messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
print("Built", len(messages_list), "message lists.")

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"messages": messages_list})
print(train_dataset)

## Model and tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded in 4-bit.")

## LoRA (PEFT) configuration

**FedAvg**: Use this exact config on all clients so state dict keys and shapes match when averaging.

In [ ]:
from peft import LoraConfig, get_peft_model

LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "./sft_humaneval_output/lora_adapters"

# Completion-only loss: mask prompt tokens (try DataCollatorForCompletionOnlyLM if available)
try:
    from trl import DataCollatorForCompletionOnlyLM
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
except ImportError:
    try:
        from trl.extras import DataCollatorForCompletionOnlyLM
        response_template = "<|im_start|>assistant\n"
        collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
    except ImportError:
        collator = None

training_args = SFTConfig(
    output_dir="./sft_humaneval_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    save_total_limit=1,
    max_seq_length=2048,
    packing=False,
)

In [ ]:
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)
if collator is not None:
    trainer_kwargs["data_collator"] = collator

trainer = SFTTrainer(**trainer_kwargs)
print("SFTTrainer created.")

In [ ]:
# Optional: quick sanity run (comment out after verifying)
# trainer.train(max_steps=2)

trainer.train()

## Save adapter (PEFT format)

In [ ]:
import os
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter and tokenizer to", ADAPTER_DIR)
print("Files:", os.listdir(ADAPTER_DIR))

## Get and print LoRA adapters (FedAvg-ready, visible as output)

In [ ]:
lora_state = {
    k: v.detach().cpu().clone()
    for k, v in model.named_parameters()
    if v.requires_grad
}

print("=== LoRA adapter structure (name -> shape) ===")
for name, tensor in lora_state.items():
    print(f"  {name}: {tensor.shape}")

print("\n=== Per-parameter summary (min, max, norm) ===")
for name, tensor in lora_state.items():
    t = tensor.float()
    print(f"  {name}: min={t.min().item():.4f}, max={t.max().item():.4f}, norm={t.norm().item():.4f}")

total_params = sum(p.numel() for p in lora_state.values())
print(f"\n=== Summary ===")
print(f"  Number of LoRA parameters: {len(lora_state)}")
print(f"  Total elements: {total_params}")
print("  Parameter names (for FedAvg):", list(lora_state.keys()))

**FedAvg**: Use `lora_state_dict.pt` below for aggregation. Load it with `torch.load(...)` and average the tensors with other clients' LoRA state dicts (same keys and shapes).

In [ ]:
lora_pt_path = os.path.join(ADAPTER_DIR, "lora_state_dict.pt")
torch.save(lora_state, lora_pt_path)
print("Saved FedAvg-ready LoRA state dict to", lora_pt_path)
print("File size (MB):", os.path.getsize(lora_pt_path) / (1024 * 1024))

# Optional: copy to Google Drive to keep after session
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r ./sft_humaneval_output /content/drive/MyDrive/